# TAE-IA · Módulo 6 · L24–L25 — Audio + Vision App: **Accessible Narrator**

A finished multimodal app. Upload a photo or an audio clip: it works out what is there and
**says it out loud**. The audio half identifies a sound or transcribes speech; the vision half
counts objects and writes a caption; both hand their sentence to the same narration stage.

| | |
|---|---|
| **Audio half** | Sections 1–6 — models, the input contract, the pipeline, its tests, and the `gr.Audio` trap |
| **Vision half** | Sections 7–8 — YOLO + BLIP, and the join |
| **The app** | Section 9 — two tabs, one narration stage |
| **GPU** | T4 (Colab) |
| **Models** | ESC-50 checkpoint 20 MB · Whisper small 0.5 GB · XTTS-v2 1.8 GB · MusicGen-small on demand · YOLOv8n 6 MB · BLIP-large 1.9 GB |

*Runtime > Run all* runs it top to bottom and ends with the app live. On a fresh runtime the
models take about 4 minutes to download.

### What it needs on Drive

Everything lives in `MyDrive/TAE_IA_M6/`:

- **`ESC50_best.pth`** — the ESC-50 classifier trained in L17. Section 2 stops without it.
- **`ESC-50-master/meta/esc50.csv`** — the class names. Without it the labels fall back to
  `class-37` instead of `rain`.
- **`inputs/`** — the shared clips and photos from L11/L12 and L24. `list_inputs()` shows what is
  there and `upload_inputs()` adds more. Nothing here downloads test files from the internet.

---

## 1 — Setup and model cache
> Weights on the runtime disk, outputs on Drive. Re-run this every session.

In [ ]:
import os, sys, gc, time, random, textwrap
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'
OUTPUT_DIR = f'{DRIVE_ROOT}/L24_output'

# One inputs folder, shared by every lab in the module - the same one L11/L12
# uses. Clips and images live here so no lesson depends on a website being up.
INPUT_DIR  = f'{DRIVE_ROOT}/inputs'

for d in (OUTPUT_DIR, INPUT_DIR):
    os.makedirs(d, exist_ok=True)

# Models live on the runtime disk: fast, roomy, symlink-safe.
# Wiped when the runtime is recycled -> ~4 min re-download each session.
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']          = MODEL_CACHE
os.environ['TORCH_HOME']       = MODEL_CACHE
os.environ['XDG_CACHE_HOME']   = MODEL_CACHE
os.environ['TTS_HOME']         = f'{MODEL_CACHE}/tts'   # where coqui-tts puts XTTS
os.environ['YOLO_CONFIG_DIR']  = MODEL_CACHE            # the vision half; keeps YOLO off $HOME
os.environ['COQUI_TOS_AGREED'] = '1'           # MUST be set before `import TTS`

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('empty')

In [ ]:
# `TTS` (the original Coqui package) caps at Python < 3.12 and cannot install on Colab any
# more. `coqui-tts` is the maintained fork; the [codec] extra pulls in torchcodec, which
# torch >= 2.9 needs for audio I/O. Same line L21 and L22 use.
# ultralytics is the vision half's YOLO - installed here so the rest of the notebook never
# stops to pip.
!pip install -q openai-whisper ultralytics gradio librosa soundfile
!pip install -q "coqui-tts[codec]"
# Measured on Colab 2026-09-18: huggingface_hub stays at 1.29.0, pip check is clean, and the
# vision half's snapshot_download + BLIP work fine afterwards. Do not upgrade hub after this
# install: that is what breaks `import TTS`.

## 2 — Load the audio models
> Three resident, one on demand. Watch the VRAM line grow, and note what we *do not* load.

In [ ]:
import librosa, soundfile as sf
import whisper
import torch.nn as nn
from torchvision import models as tvm, transforms as T
from PIL import Image

# --- 1. Your ESC-50 classifier from L17 (20 MB, ~10 ms) ---
CKPT = f'{DRIVE_ROOT}/ESC50_best.pth'
if not os.path.exists(CKPT):
    raise FileNotFoundError(
        f'{CKPT} not found. This is the one artefact you cannot re-download - '
        'it is the checkpoint you trained in L17.')

def build_classifier(num_classes=50):
    m = tvm.efficientnet_b0(weights=None)                 # weights come from the ckpt
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

clf = build_classifier().to('cuda').eval()
state = torch.load(CKPT, map_location='cuda')   # L17 saves a bare state_dict
clf.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)

# Label names: from the ESC-50 metadata if it survived, else numeric.
ESC50_META = f'{DRIVE_ROOT}/ESC-50-master/meta/esc50.csv'
if os.path.exists(ESC50_META):
    import pandas as pd
    _m = pd.read_csv(ESC50_META).drop_duplicates('target').sort_values('target')
    ESC50_LABELS = _m['category'].tolist()
else:
    ESC50_LABELS = [f'class-{i}' for i in range(50)]
    print('  ! esc50.csv not on Drive - labels will be numeric')
print(f'classifier: {len(ESC50_LABELS)} classes'); vram('after classifier')

# --- 2. Whisper small (0.5 GB download, ~2 GB VRAM, seconds per clip) ---
asr = whisper.load_model('small', download_root=MODEL_CACHE)
vram('after Whisper')

# --- 3. XTTS-v2 (1.8 GB, ~2 GB VRAM, seconds per sentence) ---
# coqui-tts 0.27 still imports a helper transformers 5.1 removed. One line, before the
# import, or `from TTS.api import ...` raises ImportError. Upstream: idiap/coqui-ai-TTS#558.
import transformers.pytorch_utils as _pu
if not hasattr(_pu, 'isin_mps_friendly'):
    _pu.isin_mps_friendly = torch.isin

from TTS.api import TTS as CoquiTTS
tts = CoquiTTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
vram('all three resident')

# --- 4. MusicGen: NOT loaded. It is ~4 GB and used on one branch only.
#        Loaded on demand in `generate()` below, and freed straight after.
#        This is the decision that lets the audio and vision halves share one T4.

## 3 — Helpers: the input contract
> `coerce_audio` is `coerce_image`'s twin. Gradio hands us `(sr, wav)` — rate first.

In [ ]:
import requests
from io import BytesIO

# The rates every stage wants. Four different numbers in one pipeline.
CLASSIFIER_SR = 22050    # L16: TARGET_SR
ASR_SR        = 16000    # Whisper resamples internally, but be explicit
TTS_SR        = 24000    # XTTS-v2 output
MAX_SECONDS   = 60       # a 30-min upload is the #1 cause of a demo-day stall
MIN_SECONDS   = 0.5

# --- Inputs come from Drive, not from the internet -------------------------
# Same shared folder as L11/L12. Upload clips once; no lesson ever depends on a
# website being reachable. (Wikimedia rate-limits a whole classroom at once.)

def list_inputs():
    """What is actually in the shared folder right now."""
    names = sorted(f for f in os.listdir(INPUT_DIR) if not f.startswith('.'))
    print(f'{INPUT_DIR}  ({len(names)} files)')
    for n in names:
        print(f'  {os.path.getsize(os.path.join(INPUT_DIR, n))/1e6:6.2f} MB  {n}')
    return names

def upload_inputs():
    """Pick files from your machine; they land in Drive and stay there."""
    from google.colab import files
    for fname, data in files.upload().items():
        with open(os.path.join(INPUT_DIR, fname), 'wb') as f:
            f.write(data)
        print(f'  saved {fname}')

def load_audio(name_or_path, sr=None):
    """A name in the inputs folder, or any path -> (wav float32 mono, sr)."""
    path = name_or_path
    if not os.path.isabs(path):
        path = os.path.join(INPUT_DIR, name_or_path)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{name_or_path} is not in {INPUT_DIR}. '
            'Run upload_inputs() and choose it, or copy it into that folder in Drive.')
    wav, got_sr = librosa.load(path, sr=sr, mono=True)   # sr=None keeps the original
    return wav.astype('float32'), got_sr

# The reference voice for XTTS: the inputs folder first, then whatever L21 wrote.
REF_WAV = next((p for p in (f'{INPUT_DIR}/reference.wav',
                            f'{DRIVE_ROOT}/tts_output/reference.wav')
                if os.path.exists(p)), None)
REF_SPEAKER = None if REF_WAV else 'Ana Florence'       # built-in studio voice
print('XTTS reference:', REF_WAV or f'built-in {REF_SPEAKER}')

def coerce_audio(wav, sr, target_sr):
    """Anything a user can hand us -> (float32 mono at target_sr), or ValueError."""
    if wav is None:
        raise ValueError('No audio received. Please upload a clip.')
    wav = np.asarray(wav)
    if wav.dtype.kind in 'iu':                       # gradio hands back int16
        wav = wav.astype('float32') / np.iinfo(wav.dtype).max
    wav = wav.astype('float32')
    if wav.ndim > 1:                                 # to mono, either layout
        wav = wav.mean(axis=0) if wav.shape[0] < wav.shape[1] else wav.mean(axis=1)
    if wav.size < sr * MIN_SECONDS:
        raise ValueError(f'Clip too short ({wav.size/sr:.2f} s). Use at least {MIN_SECONDS} s.')
    if wav.size > sr * MAX_SECONDS:
        wav = wav[:int(sr * MAX_SECONDS)]            # truncate, and the caller says so
    if np.abs(wav).max() < 1e-4:
        raise ValueError('That clip is silent. Whisper would invent words for it.')
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    return wav.astype('float32'), target_sr

# --- the L16 spectrogram, reproduced EXACTLY. Any drift here breaks the checkpoint. ---
N_FFT, HOP_LENGTH, N_MELS, FMAX, IMG_SIZE = 2048, 512, 128, 8000, 128
TARGET_SAMPLES = CLASSIFIER_SR * 5

_tf = T.Compose([
    T.Resize((224, 224)),                                  # L17 resized to 224, not 128
    T.ToTensor(),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),                 # grayscale -> 3 channels
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def mel_image(wav):
    """5 s at 22050 Hz -> the same 128x128 grayscale PIL image L16 saved as PNG."""
    y = wav[:TARGET_SAMPLES] if len(wav) > TARGET_SAMPLES else \
        np.pad(wav, (0, TARGET_SAMPLES - len(wav)))
    S    = librosa.feature.melspectrogram(y=y, sr=CLASSIFIER_SR, n_fft=N_FFT,
                                          hop_length=HOP_LENGTH, n_mels=N_MELS, fmax=FMAX)
    S_db = librosa.power_to_db(S, ref=np.max)
    S_n  = ((S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8) * 255).astype(np.uint8)
    S_n  = np.flipud(S_n)                                  # low freq at bottom
    return Image.fromarray(S_n).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)  # 2-D uint8 -> 'L'

def classify(wav, sr, topk=3):
    """(wav, sr) -> [(label, confidence)] sorted. Assumes sr == CLASSIFIER_SR."""
    assert sr == CLASSIFIER_SR, f'{sr} != {CLASSIFIER_SR} - coerce first'
    x = _tf(mel_image(wav)).unsqueeze(0).to('cuda')
    with torch.no_grad():
        p = torch.softmax(clf(x)[0], dim=-1)
    conf, idx = p.topk(topk)
    return [(ESC50_LABELS[i], float(c)) for c, i in zip(conf, idx)]

def transcribe(wav, sr, language=None):
    """(wav, sr) -> text. Whisper wants float32 at 16 kHz."""
    assert sr == ASR_SR, f'{sr} != {ASR_SR} - coerce first'
    out = asr.transcribe(wav, language=language, fp16=True)
    return out['text'].strip()

def narrate(text, language='es'):
    """text -> path to a 24 kHz WAV. The one stage both branches share."""
    out_wav = f'{OUTPUT_DIR}/narration.wav'
    kw = {'speaker_wav': REF_WAV} if REF_SPEAKER is None else {'speaker': REF_SPEAKER}
    tts.tts_to_file(text=text, language=language, file_path=out_wav, **kw)
    return out_wav

def generate(prompt, seconds=5):
    """text -> (wav, sr). Loads MusicGen, uses it, frees it. ~15 s per 10 s of audio."""
    from transformers import AutoProcessor, MusicgenForConditionalGeneration
    proc = AutoProcessor.from_pretrained('facebook/musicgen-small')
    mg   = MusicgenForConditionalGeneration.from_pretrained(
               'facebook/musicgen-small', torch_dtype=torch.float16).to('cuda')
    try:
        inp = proc(text=[prompt], padding=True, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = mg.generate(**inp, do_sample=True, max_new_tokens=int(seconds * 50))
        return out[0, 0].cpu().float().numpy(), mg.config.audio_encoder.sampling_rate
    finally:
        del mg, proc                      # the residency policy, in three lines
        gc.collect(); torch.cuda.empty_cache()

## 4 — The audio pipeline
> The conditional pattern: what we heard decides what happens next. This is the whole backend.

In [ ]:
CONF_FLOOR = 0.45          # below this we refuse to name the sound

def analyse_audio(wav, sr):
    """Detect what kind of clip this is, and describe it. Returns a dict, with timings."""
    t = {}

    # Branch on content, not on file type. Speech goes to Whisper, everything else
    # to the classifier. A cheap heuristic first, then the expensive model.
    w16, _ = coerce_audio(wav, sr, ASR_SR)
    t0 = time.time(); text = transcribe(w16, ASR_SR); t['whisper'] = time.time() - t0

    if len(text.split()) >= 3:
        kind, label, conf = 'speech', None, None
        summary = f'Escuché voz. Dice: {text}'
    else:
        w22, _ = coerce_audio(wav, sr, CLASSIFIER_SR)
        t0 = time.time(); top = classify(w22, CLASSIFIER_SR); t['classifier'] = time.time() - t0
        label, conf = top[0]
        kind = 'sound'
        # A 50-class model cannot say "not one of mine". The threshold says it for us.
        summary = (f'Este sonido parece {label}.' if conf >= CONF_FLOOR
                   else 'No pude identificar este sonido.')

    t['total'] = sum(t.values())
    return {'kind': kind, 'text': text, 'label': label, 'confidence': conf,
            'summary': summary, 'timings': t}

def run_audio(x, sr=None, language='es'):
    """The one function the UI calls. Anything in, never raises."""
    try:
        if isinstance(x, str):
            wav, sr = load_audio(x, sr=None)
        else:
            wav = x
            if sr is None:
                raise ValueError('No sample rate given. Audio is always (wav, sr).')
        wav, sr = coerce_audio(wav, sr, sr)          # validates; does not resample yet
    except ValueError as e:
        return None, None, str(e)

    try:
        a       = analyse_audio(wav, sr)
        out_wav = narrate(a['summary'], language=language)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, None, 'The GPU ran out of memory. Try a shorter clip.'
    except Exception as e:
        print(f'[run_audio] {type(e).__name__}: {e}')                  # for us
        return None, None, 'Something went wrong. Try another clip.'   # for them

    stages = ', '.join(f'{k} {v:.2f}s' for k, v in a['timings'].items() if k != 'total')
    report = (f"heard   : {a['kind']}"
              + (f" ({a['label']}, {a['confidence']:.2f})" if a['label'] else '') + '\n'
              f"said    : {a['summary']}\n"
              f"time    : {stages}")
    return a['summary'], out_wav, report

## 5 — Test the backend before it ever meets a user
> Every hostile input returns a sentence, not a traceback.

In [ ]:
# Real clips if you have put any in the shared folder; a synthetic one always.
rng   = np.random.default_rng(SEED)
noise = (0.2 * rng.standard_normal(CLASSIFIER_SR * 5)).astype('float32')

print('--- what is in the inputs folder ---')
clips = [f for f in list_inputs() if f.lower().endswith(('.wav', '.mp3', '.ogg', '.flac'))]

print('\n--- real runs ---')
for name in clips[:3]:
    wav, sr = load_audio(name)
    _s, _w, report = run_audio(wav, sr=sr)
    print(f'{name}:'); print(report)

if not clips:
    print('(no clips in the folder - upload_inputs() to add some)')
summary, wav_path, report = run_audio(noise, sr=CLASSIFIER_SR)
print('synthetic noise:'); print(report)

# Hostile inputs: none of these may raise.
print('\n--- hostile inputs ---')
cases = [
    ('None',        None,                                              CLASSIFIER_SR),
    ('silence',     np.zeros(CLASSIFIER_SR * 5, dtype='float32'),      CLASSIFIER_SR),
    ('stereo',      np.stack([noise, noise], axis=-1),                 CLASSIFIER_SR),
    ('int16',       (noise * 32767).astype('int16'),                   CLASSIFIER_SR),
    ('wrong sr',    noise,                                             44100),
    ('0.1 s',       noise[:int(CLASSIFIER_SR * 0.1)],                  CLASSIFIER_SR),
    ('30 minutes',  np.tile(noise, 360),                               CLASSIFIER_SR),
]
for tag, bad, sr in cases:
    _s, _w, msg = run_audio(bad, sr=sr)
    print(f'  {tag:12s} -> {msg.splitlines()[0][:64]}')

vram('peak')
print(f"peak {torch.cuda.max_memory_allocated()/1e9:.2f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print('\nThis is the VRAM the vision half has left to work with.')

## 6 — Hello Gradio, and the trap — audio edition
> Same lesson as L12, new type. Look at what it actually handed us before trusting it.

In [ ]:
import gradio as gr
print('gradio', gr.__version__)

try:
    demo.close()        # re-running this cell: stop the previous app first, or it keeps its port
except NameError:
    pass

def what_did_i_get(audio):
    if audio is None:
        return 'Got None.'
    a, b = audio
    return (f'tuple of ({type(a).__name__}, {type(b).__name__})\n'
            f'  first  = {a}  <- this is the SAMPLE RATE\n'
            f'  second = ndarray shape={b.shape} dtype={b.dtype}\n\n'
            'librosa.load returns (wav, sr). Gradio returns (sr, wav).\n'
            'Unpack it backwards and nothing raises - you just resample to the\n'
            'array length and hear noise.')

demo = gr.Interface(fn=what_did_i_get,
                    inputs=gr.Audio(type='numpy'),
                    outputs=gr.Textbox(lines=8),
                    title='What did I just get?')
demo.launch(share=True, quiet=True)

In [ ]:
# Stop this app when you are done with it. Section 9's app runs next to it on its own port.
# demo.close()

---
# Part 2 — the vision half, and the finished app

## 7 — The vision half
> The same backend L11 built, loaded next to the audio models so both halves share one runtime.
> **Two vision models, each doing something different:** YOLO counts *what* is there and where;
> BLIP describes the scene. That is the final project's first entry condition, met by this app.

In [ ]:
from ultralytics import YOLO
from transformers import BlipProcessor, BlipForConditionalGeneration
from huggingface_hub import snapshot_download

os.environ['YOLO_CONFIG_DIR'] = MODEL_CACHE

yolo = YOLO('yolov8n.pt')                                  # 6 MB
blip_dir = os.path.join(MODEL_CACHE, 'blip-large')
snapshot_download('Salesforce/blip-image-captioning-large', local_dir=blip_dir,
                  allow_patterns=['*.json', '*.txt', '*.safetensors'])
blip_proc  = BlipProcessor.from_pretrained(blip_dir)
blip_model = BlipForConditionalGeneration.from_pretrained(
    blip_dir, torch_dtype=torch.float16).to('cuda').eval()
vram('audio + vision resident')

MAX_SIDE, MIN_SIDE = 1024, 32

def coerce_image(x):
    """The L11 contract, unchanged: anything a user can hand us -> PIL RGB."""
    if x is None:
        raise ValueError('No image received. Please upload one.')
    if isinstance(x, np.ndarray):
        x = Image.fromarray(x.astype(np.uint8))
    if not isinstance(x, Image.Image):
        raise ValueError('Unsupported input. Please upload an image file.')
    x = x.convert('RGB')
    w, h = x.size
    if min(w, h) < MIN_SIDE:
        raise ValueError(f'Image too small ({w}x{h}).')
    if max(w, h) > MAX_SIDE:
        s = MAX_SIDE / max(w, h)
        x = x.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return x

def caption_image(img, max_new_tokens=40):
    inp = blip_proc(images=img.convert('RGB'), return_tensors='pt').to('cuda', torch.float16)
    with torch.no_grad():
        out = blip_model.generate(**inp, max_new_tokens=max_new_tokens, num_beams=3)
    return blip_proc.decode(out[0], skip_special_tokens=True).strip()

from collections import Counter
DET_CONF = 0.4             # below this YOLO keeps quiet about a box

def detect_objects(img, conf=DET_CONF):
    """PIL RGB -> (Counter of COCO labels, PIL image with the boxes drawn)."""
    r = yolo(img, conf=conf, verbose=False)[0]
    counts = Counter(yolo.names[int(c)] for c in r.boxes.cls)
    boxed  = Image.fromarray(r.plot()[:, :, ::-1])   # plot() draws in BGR
    return counts, boxed

def count_phrase(counts):
    """Counter({'person': 2, 'dog': 1}) -> '2 persons, 1 dog'."""
    if not counts:
        return 'no objects from my 80 classes'
    return ', '.join(f'{n} {k}' + ('s' if n > 1 and not k.endswith('s') else '')
                     for k, n in counts.most_common())

# Measured locally on the L25 image pack: ~0.1 s per photo. Dog, car, clock, bird and person
# are found; the chainsaw comes back as 'bench' and the vacuum as 'airplane' - neither is one
# of COCO's 80 classes. Same lesson as CLAP's label list: a closed vocabulary cannot say 'none'.

## 8 — The join
> Short, because both halves were built to the same shape. That is the whole argument.
> The image branch now joins **two vision models** before it narrates: YOLO's counts + BLIP's caption.

In [ ]:
def describe_image(image, language='es'):
    """Image in -> (text, spoken text, image with boxes). YOLO + BLIP, then the shared narrate stage."""
    try:
        img = coerce_image(image)
    except ValueError as e:
        return str(e), None, None
    counts, boxed = detect_objects(img)          # vision model 1: what, how many, where
    caption       = caption_image(img)           # vision model 2: the scene in words
    text = f'I can see {count_phrase(counts)}. {caption}.'
    return text, narrate(text, language=language), boxed

def describe_audio(audio, language='es'):
    """Audio in -> what we heard + spoken summary. Gradio hands back (sr, wav): rate FIRST."""
    if audio is None:
        return 'No audio received. Please upload a clip.', None
    sr, wav = audio                       # <- the trap. Not (wav, sr).
    summary, out_wav, _report = run_audio(wav, sr=sr, language=language)
    return summary, out_wav

# Smoke-test both branches before Gradio is anywhere near this.
# The image comes from the same shared folder L11/L12 seeded.
test_img = next((os.path.join(INPUT_DIR, f) for f in ('street.jpg', 'people.jpg')
                 if os.path.exists(os.path.join(INPUT_DIR, f))), None)
print(describe_audio((CLASSIFIER_SR, noise))[0])
print(describe_image(Image.open(test_img) if test_img
                     else Image.new('RGB', (256, 256), 'steelblue'))[0])
vram('after both branches')

## 9 — The real app: two tabs, one narration stage
> Declare the components, then wire the events. `inputs`/`outputs` match by ORDER.

In [ ]:
def image_tab(image, language, progress=gr.Progress()):
    progress(0.3, desc='Describing...')
    text, wav, boxed = describe_image(image, language)
    progress(1.0, desc='Done')
    return text, wav, boxed

def audio_tab(audio, language, progress=gr.Progress()):
    progress(0.3, desc='Listening...')
    text, wav = describe_audio(audio, language)
    progress(1.0, desc='Done')
    return text, wav

try:
    app.close()         # re-running this cell: stop the previous app first, or it keeps its port
except NameError:
    pass

with gr.Blocks(title='Accessible Narrator') as app:
    gr.Markdown('# Accessible Narrator\n'
                'Upload an image or an audio clip. It works out what is there '
                'and says it out loud.')
    lang = gr.Dropdown(['es', 'en', 'fr', 'de', 'pt'], value='es',
                       label='Narration language')

    with gr.Tab('Image'):
        with gr.Row():
            with gr.Column():
                img_in  = gr.Image(type='pil', label='Your photo')   # PIL: the L11 contract
                img_go  = gr.Button('Describe', variant='primary')
            with gr.Column():
                img_box = gr.Image(label='What YOLO found')
                img_txt = gr.Textbox(label='Objects + caption', lines=3)
                img_wav = gr.Audio(label='Narration')
        img_go.click(image_tab, [img_in, lang], [img_txt, img_wav, img_box])

    with gr.Tab('Audio'):
        with gr.Row():
            with gr.Column():
                aud_in  = gr.Audio(type='numpy', label='Your clip')  # (sr, wav): the L24 contract
                aud_go  = gr.Button('Identify', variant='primary')
            with gr.Column():
                aud_txt = gr.Textbox(label='What we heard', lines=3)
                aud_wav = gr.Audio(label='Narration')
        aud_go.click(audio_tab, [aud_in, lang], [aud_txt, aud_wav])

app.queue().launch(share=True, quiet=True)

In [ ]:
# Stop the app when you are done with it.
# app.close()

---
## 10 — From this app to yours

This app **describes**: BLIP writes a caption, the ESC-50 classifier names a sound, and XTTS says
either one out loud. The app in your L25 skeleton **matches**: CLAP (audio) and CLIP (image) each
tag their input against *one vocabulary you wrote*, and the join is whether the two agree.

They share no functions, and their return shapes differ on purpose. What carries over is the idea:

| In this app | In yours |
|---|---|
| YOLO counts objects, BLIP writes a caption | CLIP scores the image against your vocabulary |
| ESC-50 names the sound | CLAP scores the clip against the same vocabulary |
| The join shares a *narration* stage | The join shares a *vocabulary* |
| Two tabs, two separate answers | One comparison: do sound and image agree? |

In both, the join is short because both halves were built to the same contract.

---
## Cleanup — optional

Lists what is in `MyDrive/TAE_IA_M6/` and how much space each folder takes. With `CONFIRM = True`
it also deletes the ESC-50 corpus and the spectrogram PNGs, both re-derivable. The checkpoint is
never touched. With the default `CONFIRM = False` it only lists.

In [ ]:
import shutil

def gb(path):
    if not os.path.exists(path):
        return 0.0
    return sum(os.path.getsize(os.path.join(d, f))
               for d, _, fs in os.walk(path) for f in fs) / 1e9

print(f'{DRIVE_ROOT}  ({gb(DRIVE_ROOT):.2f} GB)')
for e in sorted(os.listdir(DRIVE_ROOT)):
    p = os.path.join(DRIVE_ROOT, e)
    print(f'  {gb(p) if os.path.isdir(p) else os.path.getsize(p)/1e9:6.2f} GB  {e}')

# Safe to delete: the raw ESC-50 corpus and the derived PNGs. Both are re-derivable,
# and together they are usually the largest thing on the Drive.
CANDIDATES = [f'{DRIVE_ROOT}/ESC-50-master', f'{DRIVE_ROOT}/ESC50_specs']

CONFIRM = False          # set True and re-run to delete
if CONFIRM:
    for p in CANDIDATES:
        if os.path.exists(p):
            freed = gb(p); shutil.rmtree(p)
            print(f'Freed {freed:.2f} GB from {p}')
else:
    print('\nWould delete:', [p for p in CANDIDATES if os.path.exists(p)])
    print('Review, then set CONFIRM = True and re-run.')

# These must survive.
print('\ncheckpoint kept:', os.path.exists(CKPT))
print('outputs kept   :', os.path.exists(OUTPUT_DIR))

---
*TAE-IA 2025 · Cocyten-Nayarit · Módulo 6 · L24–L25*